# Sesion 11: Object Tracking con YOLOv11 + ByteTrack

Guido Anthony Chipana Calderon

**Contenido:**
- Configuracion de entorno
- Tracking basico con modelo pre-entrenado
- Configuracion de parametros de ByteTrack
- Analisis de resultados (IDs unicos, trayectorias)
- Visualizacion de metricas

**Nota:** NO se requiere entrenar modelo. YOLOv11 pre-entrenado ya incluye capacidad de tracking.

## 1. Configuracion Inicial

In [1]:
# Verificar GPU
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU disponible: False


In [2]:
# Instalar Ultralytics (incluye YOLOv11 + ByteTrack)
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.2 MB/s eta 0:00:00


In [3]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Configuracion de Rutas

In [4]:
import os

# Rutas base
BASE_DIR = '/content/drive/MyDrive/curso_cv'
DATASETS_DIR = os.path.join(BASE_DIR, 'datasets')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
OUTPUTS_DIR = os.path.join(BASE_DIR, 'outputs')
SESSION_OUTPUT = os.path.join(OUTPUTS_DIR, 'sesion11')

# Crear carpeta de salida si no existe
os.makedirs(SESSION_OUTPUT, exist_ok=True)

print("Estructura de directorios:")
print(f"Base: {BASE_DIR}")
print(f"Datasets: {DATASETS_DIR}")
print(f"Models: {MODELS_DIR}")
print(f"Output Sesion 11: {SESSION_OUTPUT}")

Estructura de directorios:
Base: /content/drive/MyDrive/curso_cv
Datasets: /content/drive/MyDrive/curso_cv/datasets
Models: /content/drive/MyDrive/curso_cv/models
Output Sesion 11: /content/drive/MyDrive/curso_cv/outputs/sesion11


In [5]:
VIDEO_NAME = 'video_test_sesion8.mp4'

VIDEO_PATH = os.path.join(DATASETS_DIR, VIDEO_NAME)

# Verificar que existe
if os.path.exists(VIDEO_PATH):
    print(f"Video encontrado: {VIDEO_PATH}")
else:
    print(f"ERROR: No se encuentra el video {VIDEO_PATH}")
    print("Por favor verifica el nombre del archivo.")

Video encontrado: /content/drive/MyDrive/curso_cv/datasets/video_test_sesion8.mp4


## 3. Cargar Modelo YOLOv11

**Importante:** Usamos modelo pre-entrenado. NO requiere entrenamiento adicional para tracking.

In [6]:
from ultralytics import YOLO

# Cargar modelo YOLOv11 nano (rapido y eficiente)
# Opciones: yolov11n.pt (nano), yolov11s.pt (small), yolov11m.pt (medium)
model = YOLO(f'{OUTPUTS_DIR}/sesion8/B_backbone_congelado/weights/best.pt')

print("Modelo cargado correctamente")
print(f"Clases disponibles: {len(model.names)}")
print(f"Ejemplo de clases: {list(model.names.values())[:10]}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Modelo cargado correctamente
Clases disponibles: 1
Ejemplo de clases: ['persona']


## 4. Tracking Basico

Tracking simple con parametros por defecto.

In [8]:
# Tracking basico
OUTPUT_VIDEO_BASIC = os.path.join(SESSION_OUTPUT, 'tracking_persona.mp4')

results = model.track(
    source=VIDEO_PATH,
    conf=0.5,              # Confianza minima para detectar
    iou=0.5,               # IoU threshold para NMS
    show=False,            # No mostrar en pantalla
    save=True,             # Guardar video con tracking
    tracker="bytetrack.yaml",
    project=SESSION_OUTPUT,
    name='tracking_persona',
    exist_ok=True
)

print("Tracking basico completado")
print(f"Video guardado en: {SESSION_OUTPUT}/tracking_basico")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 146ms
Prepared 1 package in 34ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/345) /content/drive/MyDrive/curso_cv/datasets/video_test_sesion8.mp4: 384x640 40 p

## 5. Tracking con Parametros Personalizados

Configuracion avanzada de ByteTrack para mejor rendimiento.

In [9]:
# Crear archivo de configuracion ByteTrack personalizado con parametros completos
BYTETRACK_CONFIG = os.path.join(SESSION_OUTPUT, 'bytetrack_custom.yaml')

# Agregamos fuse_score y gmc_method para evitar el AttributeError
config_content = """tracker_type: bytetrack

# Umbrales de confianza
track_high_thresh: 0.5    # Detecciones de alta confianza
track_low_thresh: 0.1     # Detecciones de baja confianza (para recuperar objetos)

# Asociacion
match_thresh: 0.8         # IoU minimo para asociar deteccion con track
fuse_score: True          # Obligatorio para evitar AttributeError

# Persistencia
track_buffer: 30          # Frames que mantener track sin deteccion

# Nuevos tracks
new_track_thresh: 0.6     # Confianza minima para crear nuevo track

# Motion analysis
gmc_method: kalman        # Metodo de compensacion de movimiento
"""

with open(BYTETRACK_CONFIG, 'w') as f:
    f.write(config_content)

print("Configuracion ByteTrack corregida:")
print(config_content)

Configuracion ByteTrack corregida:
tracker_type: bytetrack

# Umbrales de confianza
track_high_thresh: 0.5    # Detecciones de alta confianza
track_low_thresh: 0.1     # Detecciones de baja confianza (para recuperar objetos)

# Asociacion
match_thresh: 0.8         # IoU minimo para asociar deteccion con track
fuse_score: True          # Obligatorio para evitar AttributeError

# Persistencia
track_buffer: 30          # Frames que mantener track sin deteccion

# Nuevos tracks
new_track_thresh: 0.6     # Confianza minima para crear nuevo track

# Motion analysis
gmc_method: kalman        # Metodo de compensacion de movimiento



In [10]:
# Tracking con configuracion personalizada
results_custom = model.track(
    source=VIDEO_PATH,
    conf=0.3,
    iou=0.5,
    tracker=BYTETRACK_CONFIG,  # Usar config personalizada
    show=False,
    save=True,
    project=SESSION_OUTPUT,
    name='tracking_personalizado',
    exist_ok=True,
    classes=[0]
)

print("Tracking personalizado completado")
print(f"Video guardado en: {SESSION_OUTPUT}/tracking_personalizado")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/345) /content/drive/MyDrive/curso_cv/datasets/video_test_sesion8.mp4: 384x640 40 personas, 135.6ms
video 1/1 (frame 2/345) /content/drive/MyDrive/curso_cv/datasets/video_test_sesion8.mp4: 384x640 40 personas, 136.3ms
video 1/1 (frame 3/345) /content/drive/MyDrive/curso_cv/datasets/video_test_sesion8.mp4: 384x640 40 personas, 139.7ms
video 1/1 (frame 4/345) /content/drive/MyDrive/curso_cv/datasets/video_test_sesion8.mp4: 384x640 40 pers

## 6. Analisis de Resultados

Extraer metricas del tracking: IDs unicos, duracion de tracks, etc.

In [11]:
import cv2
from collections import defaultdict

# Procesar video frame por frame para analisis detallado
print("Analizando tracking frame por frame...")

# Diccionarios para almacenar metricas
track_ids = set()
track_lifetimes = defaultdict(int)  # Frames que aparece cada ID
track_first_seen = {}               # Primer frame donde aparece cada ID
track_last_seen = {}                # Ultimo frame donde aparece cada ID

# Procesar video
results_analysis = model.track(
    source=VIDEO_PATH,
    conf=0.3,
    tracker=BYTETRACK_CONFIG,
    stream=True,  # Procesar frame por frame
    verbose=False,
    classes=[0]   # Solo personas
)

frame_count = 0
for result in results_analysis:
    frame_count += 1

    if result.boxes.id is not None:
        ids = result.boxes.id.cpu().numpy().astype(int)

        for track_id in ids:
            track_ids.add(track_id)
            track_lifetimes[track_id] += 1

            if track_id not in track_first_seen:
                track_first_seen[track_id] = frame_count

            track_last_seen[track_id] = frame_count

print(f"Analisis completado: {frame_count} frames procesados")

Analizando tracking frame por frame...
Analisis completado: 341 frames procesados


In [12]:
# Mostrar estadisticas
print("\nESTADISTICAS DE TRACKING")
print(f"Total frames procesados: {frame_count}")
print(f"Total objetos unicos detectados: {len(track_ids)}")
print(f"\nTop 10 tracks mas duraderos:")

# Ordenar por duracion
sorted_lifetimes = sorted(track_lifetimes.items(), key=lambda x: x[1], reverse=True)

print("\nID | Frames | Duracion (frames) | Primero | Ultimo")
for track_id, lifetime in sorted_lifetimes[:10]:
    first = track_first_seen[track_id]
    last = track_last_seen[track_id]
    duration = last - first + 1
    print(f"{track_id:3d} | {lifetime:6d} | {duration:17d} | {first:7d} | {last:6d}")

# Estadisticas agregadas
import numpy as np
lifetimes_list = list(track_lifetimes.values())
print("\nMETRICAS AGREGADAS")
print(f"Duracion promedio de track: {np.mean(lifetimes_list):.1f} frames")
print(f"Duracion mediana: {np.median(lifetimes_list):.1f} frames")
print(f"Duracion maxima: {np.max(lifetimes_list)} frames")
print(f"Duracion minima: {np.min(lifetimes_list)} frames")


ESTADISTICAS DE TRACKING
Total frames procesados: 341
Total objetos unicos detectados: 76

Top 10 tracks mas duraderos:

ID | Frames | Duracion (frames) | Primero | Ultimo
  4 |    341 |               341 |       1 |    341
 11 |    341 |               341 |       1 |    341
 15 |    341 |               341 |       1 |    341
 17 |    341 |               341 |       1 |    341
 22 |    341 |               341 |       1 |    341
 30 |    341 |               341 |       1 |    341
 33 |    339 |               341 |       1 |    341
 19 |    332 |               333 |       1 |    333
 38 |    318 |               341 |       1 |    341
 16 |    315 |               341 |       1 |    341

METRICAS AGREGADAS
Duracion promedio de track: 161.0 frames
Duracion mediana: 137.0 frames
Duracion maxima: 341 frames
Duracion minima: 1 frames


## 7. Visualizacion de Trayectorias

Dibujar trayectorias completas de cada objeto detectado.

In [13]:
import cv2
import numpy as np
from collections import defaultdict

# Diccionario para guardar trayectorias
track_history = defaultdict(lambda: [])

# Abrir video para escribir resultado
cap = cv2.VideoCapture(VIDEO_PATH)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

OUTPUT_TRAJ = os.path.join(SESSION_OUTPUT, 'tracking_trayectorias.mp4')
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_TRAJ, fourcc, fps, (width, height))

print(f"Procesando video con trayectorias...")
print(f"Resolucion: {width}x{height} @ {fps} FPS")

# Procesar video
results_traj = model.track(
    source=VIDEO_PATH,
    conf=0.3,
    tracker=BYTETRACK_CONFIG,
    stream=True,
    verbose=False,
    classes=[0]
)

frame_idx = 0
for result in results_traj:
    frame_idx += 1
    frame = result.orig_img.copy()

    # Obtener detecciones con IDs
    if result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.cpu().numpy().astype(int)

        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = map(int, box)

            # Centro del bounding box
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            # Guardar punto en trayectoria
            track_history[track_id].append((cx, cy))

            # Mantener solo ultimos 30 puntos
            if len(track_history[track_id]) > 30:
                track_history[track_id].pop(0)

            # Dibujar bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Dibujar ID
            cv2.putText(frame, f'ID: {track_id}', (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            # Dibujar trayectoria
            points = track_history[track_id]
            for i in range(1, len(points)):
                # Color que se desvanece
                alpha = i / len(points)
                thickness = max(1, int(3 * alpha))
                cv2.line(frame, points[i-1], points[i],
                        (255, 0, 0), thickness)

    # Agregar info del frame
    cv2.putText(frame, f'Frame: {frame_idx}', (10, 30),
               cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    # Escribir frame
    out.write(frame)

    if frame_idx % 30 == 0:
        print(f"Procesados {frame_idx} frames...")

# Liberar recursos
cap.release()
out.release()

print(f"\nVideo con trayectorias guardado en: {OUTPUT_TRAJ}")

Procesando video con trayectorias...
Resolucion: 1280x720 @ 25 FPS
Procesados 30 frames...
Procesados 60 frames...
Procesados 90 frames...
Procesados 120 frames...
Procesados 150 frames...
Procesados 180 frames...
Procesados 210 frames...
Procesados 240 frames...
Procesados 270 frames...
Procesados 300 frames...
Procesados 330 frames...

Video con trayectorias guardado en: /content/drive/MyDrive/curso_cv/outputs/sesion11/tracking_trayectorias.mp4


## 8. Conteo con Linea Virtual

Implementar conteo de objetos que cruzan una linea virtual.

In [14]:
# Definir linea virtual
# Formato: (x1, y1, x2, y2)
LINE_START = (width // 4, height // 2)    # Punto inicial
LINE_END = (3 * width // 4, height // 2)  # Punto final

print(f"Linea virtual definida: {LINE_START} -> {LINE_END}")

Linea virtual definida: (320, 360) -> (960, 360)


In [15]:
def ccw(A, B, C):
    """Determina orientacion de tres puntos (counter-clockwise)"""
    return (C[1] - A[1]) * (B[0] - A[0]) > (B[1] - A[1]) * (C[0] - A[0])

def intersect(A, B, C, D):
    """Verifica si segmento AB intersecta segmento CD"""
    return ccw(A, C, D) != ccw(B, C, D) and ccw(A, B, C) != ccw(A, B, D)

# Contadores
count_up = 0      # Objetos que cruzan hacia arriba
count_down = 0    # Objetos que cruzan hacia abajo
crossed_ids = set()  # IDs que ya cruzaron (para no contar dos veces)

# Posiciones anteriores de cada track
previous_positions = {}

# Procesar video
cap = cv2.VideoCapture(VIDEO_PATH)
OUTPUT_COUNT = os.path.join(SESSION_OUTPUT, 'tracking_conteo.mp4')
out = cv2.VideoWriter(OUTPUT_COUNT, fourcc, fps, (width, height))

results_count = model.track(
    source=VIDEO_PATH,
    conf=0.3,
    tracker=BYTETRACK_CONFIG,
    stream=True,
    verbose=False,
    classes=[0]
)

print("Procesando conteo con linea virtual...")

for result in results_count:
    frame = result.orig_img.copy()

    # Dibujar linea virtual
    cv2.line(frame, LINE_START, LINE_END, (0, 0, 255), 3)

    if result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.cpu().numpy().astype(int)

        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = map(int, box)
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            current_pos = (cx, cy)

            # Si tenemos posicion anterior, verificar cruce
            if track_id in previous_positions and track_id not in crossed_ids:
                prev_pos = previous_positions[track_id]

                # Verificar si cruzo la linea
                if intersect(prev_pos, current_pos, LINE_START, LINE_END):
                    # Determinar direccion
                    if current_pos[1] < prev_pos[1]:
                        count_up += 1
                        direction = 'UP'
                    else:
                        count_down += 1
                        direction = 'DOWN'

                    crossed_ids.add(track_id)
                    print(f"ID {track_id} cruzo hacia {direction}")

            # Actualizar posicion
            previous_positions[track_id] = current_pos

            # Dibujar deteccion
            color = (0, 255, 0) if track_id not in crossed_ids else (255, 0, 255)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f'ID: {track_id}', (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Mostrar contadores
    cv2.putText(frame, f'Arriba: {count_up}', (10, 40),
               cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f'Abajo: {count_down}', (10, 80),
               cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f'Total: {count_up + count_down}', (10, 120),
               cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)

    out.write(frame)

cap.release()
out.release()

print(f"\nConteo completado:")
print(f"  Cruces hacia arriba: {count_up}")
print(f"  Cruces hacia abajo: {count_down}")
print(f"  Total: {count_up + count_down}")
print(f"\nVideo guardado en: {OUTPUT_COUNT}")

Procesando conteo con linea virtual...
ID 7 cruzo hacia UP
ID 2 cruzo hacia UP
ID 5 cruzo hacia DOWN
ID 19 cruzo hacia UP
ID 8 cruzo hacia DOWN
ID 12 cruzo hacia DOWN
ID 43 cruzo hacia UP
ID 46 cruzo hacia UP
ID 25 cruzo hacia DOWN
ID 48 cruzo hacia UP
ID 27 cruzo hacia DOWN
ID 41 cruzo hacia DOWN
ID 60 cruzo hacia UP

Conteo completado:
  Cruces hacia arriba: 7
  Cruces hacia abajo: 6
  Total: 13

Video guardado en: /content/drive/MyDrive/curso_cv/outputs/sesion11/tracking_conteo.mp4


## Notas Finales

**Archivos generados:**
1. `tracking_basico/` - Video con tracking usando parametros por defecto
2. `tracking_personalizado/` - Video con configuracion ByteTrack personalizada
3. `tracking_trayectorias.mp4` - Video con trayectorias visualizadas
4. `tracking_conteo.mp4` - Video con conteo usando linea virtual
7. `bytetrack_custom.yaml` - Configuracion ByteTrack personalizada

**Parametros ajustables:**
- `track_high_thresh`: Aumentar para menos false positives
- `track_low_thresh`: Reducir si hay muchos ID switches
- `track_buffer`: Aumentar para mantener IDs mas tiempo durante oclusiones
- `match_thresh`: Ajustar segun movimiento de objetos (rapido = menor threshold)

**No se requiere entrenar modelo:**
- YOLOv11 funciona directamente con ByteTrack
- El tracker mantiene identidad de objetos detectados
- Solo ajustar parametros de tracking segun necesidad